# 05 Pipeline final y datos de demostración

Este notebook arma un Pipeline de scikit-learn que recibe las variables originales del par (sin imputar, sin escalar y con el canal y la moneda en texto), lo entrena con el mismo conjunto de entrenamiento del notebook 03, comprueba que predice lo mismo que el modelo del notebook 03 y lo guarda con joblib para la API.

Al final genera datos sintéticos con las mismas columnas, para que la API y el dashboard funcionen en el repositorio público sin datos del Banco.


In [10]:
%pip install -r ../requirements.txt

  Using cached pandas-3.0.5-cp311-cp311-win_amd64.whl.metadata (19 kB)
Note: you may need to restart the kernel to use updated packages.


ERROR: Ignored the following yanked versions: 2.4.0
ERROR: Ignored the following versions that require a different python version: 1.21.2 Requires-Python >=3.7,<3.11; 1.21.3 Requires-Python >=3.7,<3.11; 1.21.4 Requires-Python >=3.7,<3.11; 1.21.5 Requires-Python >=3.7,<3.11; 1.21.6 Requires-Python >=3.7,<3.11; 2.5.0 Requires-Python >=3.12; 2.5.0rc1 Requires-Python >=3.12; 2.5.1 Requires-Python >=3.12; 2.5.2 Requires-Python >=3.12; 2.5.3 Requires-Python >=3.12; 2.5.4 Requires-Python >=3.12; 2.5.5 Requires-Python >=3.12
ERROR: Could not find a version that satisfies the requirement numpy==2.5.3 (from versions: 1.3.0, 1.4.1, 1.5.0, 1.5.1, 1.6.0, 1.6.1, 1.6.2, 1.7.0, 1.7.1, 1.7.2, 1.8.0, 1.8.1, 1.8.2, 1.9.0, 1.9.1, 1.9.2, 1.9.3, 1.10.0.post2, 1.10.1, 1.10.2, 1.10.4, 1.11.0, 1.11.1, 1.11.2, 1.11.3, 1.12.0, 1.12.1, 1.13.0, 1.13.1, 1.13.3, 1.14.0, 1.14.1, 1.14.2, 1.14.3, 1.14.4, 1.14.5, 1.14.6, 1.15.0, 1.15.1, 1.15.2, 1.15.3, 1.15.4, 1.16.0, 1.16.1, 1.16.2, 1.16.3, 1.16.4, 1.16.5, 1.16.6, 1.17

## 1. Datos y division

Se usa la misma division estratificada 60/20/20 con random_state=42 del notebook 03. Como las filas y la etiqueta son las mismas, los tres conjuntos quedan exactamente iguales.


In [11]:
import os
import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, f1_score

pares = pd.read_parquet("../data/processed/pares.parquet")
# los que escogió el GridSearchCV en el notebook 03
hp = {"n_estimators": 200, "max_depth": None, "min_samples_leaf": 1, "class_weight": "balanced"}

columnas_continuas = ["dias_entre_facturas", "similitud_numero", "distancia_periodo", "log_monto", "log_historia_proveedor"]
columnas_binarias = ["mismo_numero_normalizado","mismo_canal","mismo_metodo"]
columnas_categoricas = ["tipo_canal", "tipo_moneda"]
columnas_dummy = [c for c in pares.columns if c.startswith("canal_") or c.startswith("moneda_")]

X = pares[columnas_continuas+columnas_binarias+columnas_categoricas+columnas_dummy]
y = pares["y"]
X_temp,X_test,y_temp,y_test = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp)
print(f"Entrenamiento: {len(X_train):,} | validación: {len(X_val):,} | prueba: {len(X_test):,}")

Entrenamiento: 6,873 | validación: 2,291 | prueba: 2,291


## 2. Pipeline

El ColumnTransformer tiene tres ramas, con los mismos pasos del notebook 03:

- Continuas: imputación con la mediana y estandarización.
- Binarias (0/1): pasan sin cambios, porque en el notebook 03 no se escalaron.
- Categoricas (canal y moneda en texto): imputacion con la moda y one-hot con drop="first" (deja csp y CAD como referencia, igual que get_dummies con drop_first=True) y handle_unknown="ignore" para que un canal nuevo no rompa la API.

El modelo es el Random Forest con los hiperparámetros que escogió GridSearchCV en el notebook 03.


In [12]:
preprocesamiento=ColumnTransformer([
    ("numericas", Pipeline([("imputacion", SimpleImputer(strategy="median")),
                            ("escalado", StandardScaler())]), columnas_continuas),
   ("binarias", "passthrough", columnas_binarias),
    ("categoricas", Pipeline([("imputacion", SimpleImputer(strategy="most_frequent")),
                           ("onehot", OneHotEncoder(drop="first", handle_unknown="ignore"))]),
   columnas_categoricas),
])
pipeline = Pipeline([
   ("preprocesamiento",preprocesamiento),
   ("modelo",RandomForestClassifier(n_estimators=hp["n_estimators"],max_depth=hp["max_depth"],
                                        min_samples_leaf=hp["min_samples_leaf"],class_weight=hp["class_weight"],
                                     random_state=42,n_jobs=-1)),
])
pipeline.fit(X_train[columnas_continuas + columnas_binarias + columnas_categoricas], y_train)
pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocesamiento', ...), ('modelo', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numericas', ...), ('binarias', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the diff

## 3. Comprobación contra el modelo del notebook 03

Se repite el procedimiento del notebook 03 sin pipeline (imputación con la mediana de entrenamiento, escalado de las continuas y columnas de get_dummies) y se comprueba con np.allclose que las dos versiones dan la misma probabilidad en prueba.


In [13]:
cols_03 = columnas_continuas + columnas_binarias + columnas_dummy
X_tr3,X_te3 = X_train[cols_03].copy(),X_test[cols_03].copy()
mediana = X_tr3["similitud_numero"].median()
X_tr3["similitud_numero"]=X_tr3["similitud_numero"].fillna(mediana)
X_te3["similitud_numero"] = X_te3["similitud_numero"].fillna(mediana)
escalador = StandardScaler()
X_tr3[columnas_continuas]=escalador.fit_transform(X_tr3[columnas_continuas])
X_te3[columnas_continuas] = escalador.transform(X_te3[columnas_continuas])
rf_03=RandomForestClassifier(n_estimators=hp["n_estimators"],max_depth=hp["max_depth"],min_samples_leaf=hp["min_samples_leaf"],
                              class_weight=hp["class_weight"],random_state=42,n_jobs=-1).fit(X_tr3,y_train)
entrada=columnas_continuas + columnas_binarias + columnas_categoricas
# debe dar True; si no, la API no daría lo mismo que el notebook
print("¿El pipeline predice lo mismo que el modelo del notebook 03?",
       np.allclose(pipeline.predict_proba(X_test[entrada])[:,1],rf_03.predict_proba(X_te3)[:,1]))

¿El pipeline predice lo mismo que el modelo del notebook 03? True


## 4. Evaluacion en validación y prueba


In [14]:
for nombre,Xc,yc in [("validación",X_val,y_val),("prueba",X_test,y_test)]:
    pred = pipeline.predict(Xc[entrada])
    print(f"{nombre:10s} precisión {precision_score(yc, pred):.3f} | recall {recall_score(yc, pred):.3f} | F1 {f1_score(yc, pred):.3f}")

validación precisión 0.719 | recall 0.189 | F1 0.299
prueba     precisión 0.776 | recall 0.240 | F1 0.366


## 5. Prediccion de un par escrito a mano

El pipeline recibe las variables originales, con texto en el canal y la moneda y un valor faltante en la similitud del número. No hay que transformar nada antes de predecir.


In [15]:
par_nuevo = pd.DataFrame([{"dias_entre_facturas": 2, "similitud_numero": np.nan, "distancia_periodo": 28, "log_monto": np.log10(45200),
                             "log_historia_proveedor": np.log1p(12), "mismo_numero_normalizado": 0, "mismo_canal": 0,
                          "mismo_metodo": 0,"tipo_canal": "email","tipo_moneda": "CAD"}])
print("Probabilidad de duplicado:", round(pipeline.predict_proba(par_nuevo)[0, 1], 4))

Probabilidad de duplicado: 0.365


## 6. Guardar el pipeline

Se guarda el pipeline completo, no solo el clasificador, para que la imputacion, el escalado y la codificación viajen con el modelo. Se imprimen las versiones, que deben coincidir con las de api/requirements.txt.


In [16]:
os.makedirs("../api/model", exist_ok=True)
joblib.dump(pipeline,"../api/model/duplicados_pipeline.joblib")
print("scikit-learn",sklearn.__version__,"| numpy",np.__version__,
    "| pandas", pd.__version__, "| joblib", joblib.__version__)

scikit-learn 1.8.0 | numpy 2.3.5 | pandas 2.3.3 | joblib 1.5.3


## 7. Datos sintéticos para la demostración

El repositorio es público y no puede llevar datos del Banco. Para que la API y el dashboard funcionen, se generan 400 pares sinteticos con las mismas columnas que recibe la API. Los valores se sortean con distribuciones simples (no se copian filas reales) y los proveedores son seudónimos inventados (DEMO_xxx). Después se puntuan con el pipeline, igual que se haría con la cola real.


In [17]:
import sys
sys.path.append("../api")
from prediction_model import calcular_variables

# datos inventados: el repo es público y no puede llevar datos del Banco
rng=np.random.default_rng(42)
n=400
canales=rng.choice(["csp", "sftp", "ui", "email", "otro"], n, p=[0.55, 0.25, 0.12, 0.05, 0.03])
cambia = rng.random(n) < 0.06
fecha_ant = pd.Timestamp("2025-09-01") + pd.to_timedelta(rng.integers(0, 240, n), unit="D")
dias=np.where(rng.random(n) < 0.6, rng.choice([28, 30, 31, 91], n), rng.integers(0, 60, n))
base_num = [f"INV{rng.integers(1000, 99999)}" for _ in range(n)]
reenvio = rng.random(n) < 0.08
demo = pd.DataFrame({
     "proveedor_id": [f"DEMO_{i:03d}" for i in rng.integers(1, 120, n)],
     "numero_factura_anterior": base_num,
   "numero_factura_nueva": [b.replace("INV","INV-") if r else f"INV{int(b[3:]) + rng.integers(1, 50)}" for b,r in zip(base_num,reenvio)],
  "fecha_factura_anterior": fecha_ant.date,
    "fecha_factura_nueva": (fecha_ant + pd.to_timedelta(np.where(reenvio, rng.integers(0, 5, n), dias), unit="D")).date,
  "canal_anterior": canales,
     "canal_nuevo": np.where(cambia,rng.choice(["csp","sftp","ui","email"],n),canales),
     "metodo_anterior": "Integration",
      "metodo_nuevo": np.where(cambia | (reenvio & (rng.random(n) < 0.5)), "Manual", "Integration"),
  "moneda": rng.choice(["CAD","USD","otra"],n,p=[0.82,0.13,0.05]),
 "monto_cad": np.round(10000 * 10 ** rng.exponential(0.45,n),2),
  "facturas_previas_proveedor": rng.integers(0, 3000, n),
})
demo["puntaje"] = pipeline.predict_proba(calcular_variables(demo))[:, 1].round(4)
os.makedirs("../api/data",exist_ok=True)
demo.to_csv("../api/data/cola_demo.csv", index=False)
print(demo["puntaje"].describe().round(3))
demo.sort_values("puntaje", ascending=False).head(5)

count    400.000
mean       0.112
std        0.146
min        0.000
25%        0.025
50%        0.055
75%        0.126
max        0.840
Name: puntaje, dtype: float64


,proveedor_id,numero_factura_anterior,numero_factura_nueva,fecha_factura_anterior,fecha_factura_nueva,canal_anterior,canal_nuevo,metodo_anterior,metodo_nuevo,moneda,monto_cad,facturas_previas_proveedor,puntaje
295,DEMO_021,INV93026,INV-93026,2026-03-24,2026-03-26,ui,ui,Integration,Manual,CAD,31337.87,1098,0.840
37,DEMO_009,INV50419,INV-50419,2026-01-28,2026-01-31,csp,csp,Integration,Manual,CAD,31520.97,1113,0.805
193,DEMO_066,INV65930,INV-65930,2025-11-06,2025-11-07,ui,ui,Integration,Manual,CAD,39069.15,865,0.790
311,DEMO_083,INV97099,INV-97099,2026-04-02,2026-04-04,csp,csp,Integration,Manual,CAD,35378.21,756,0.730
297,DEMO_039,INV2768,INV-2768,2026-03-29,2026-03-30,ui,ui,Integration,Integration,otra,87961.39,146,0.650


Para la pestaña de proveedores del dashboard se generan también grupos sintéticos de llaves con su gasto, con seudónimos inventados.


In [18]:
grupos_demo = []
for g in range(1,41):
    for k in range(rng.integers(2, 4)):
        grupos_demo.append({"grupo": f"GRUPO_D{g:02d}", "proveedor_id": f"DEMO_P{g:02d}{k}", "entidad": rng.choice(["10001", "40048"]),
                            "similitud_nombre": round(float(rng.uniform(0.90, 1.0)), 3),
                             "gasto_aprobado": round(float(10 ** rng.normal(6.5,0.6)),2)})
os.makedirs("../dashboard/data", exist_ok=True)
pd.DataFrame(grupos_demo).to_csv("../dashboard/data/proveedores_demo.csv", index=False)
print("Grupos sintéticos:",len(set(r["grupo"] for r in grupos_demo)),"| llaves:",len(grupos_demo))

Grupos sintéticos: 40 | llaves: 101
